# Linear-Quadratic Validation Framework

### Model
Let $X_0 \sim \mathcal{N}(\mu_0,\Sigma_0)$, $m_t = \mathbb{P}_{X_t}$, $\bar m_t = \mathbb{E}_{X\sim m_t}[X]$, and
\begin{equation}
X_{t+1} = aX_t + b\alpha_t + c\bar m_t + \sigma\epsilon_{t+1}, \qquad \epsilon_{t+1}\overset{\text{i.i.d.}}{\sim}\mathcal{N}(0,1).
\end{equation}
In this case, we have $P_t(\cdot | x,\alpha,m) := \mathcal{N}(ax + b\alpha + c \bar{m}, \sigma^2)$ for any $(t,x,\alpha,m) \in \mathcal{T} \backslash \lbrace N \rbrace \times \mathcal{X} \times A \times \mathcal{P}(\mathcal{X})$, and $\pi_t^\theta(\cdot | x,m) := \mathcal{N}(\theta_t^1  x + \theta_t^2 \bar{m} , \tau^2) \in \mathcal{P}(A)$.
We restrict to Gaussian feedback policies $\alpha_t = \theta_t^1 X_t + \theta_t^2 \bar m_t + \tau\eta_t$, $\eta_t\overset{\text{i.i.d.}}{\sim}\mathcal{N}(0,1)$, with running and terminal costs
\begin{equation}
r_t(x,m,a) = qx^2 + ra^2 + \gamma \bar m^2, \qquad g(x,m) = q_T x^2 + \gamma_T \bar m^2.
\end{equation}
We will optimize the full sequence
\begin{align}
    \theta = (\theta_t^1, \theta_t^2)_{t=0}^{T-1}.
\end{align}

### Randomized Perturbation
Following the perturbation framework, we apply $T^\lambda(u,x,v) = (u, x+\lambda v)$ to the mean-field at each step, with $v$ drawn from i.i.d.\ affine perturbations $U_t(x) = A_tx+B_t$, $(A_t,B_t)\overset{\text{i.i.d.}}{\sim}\mathcal{N}(0,\rho^2)\otimes\mathcal{N}(0,\rho^2)$. Since $m_t=\mathcal{N}(\mu_t,\Sigma_t)$, the perturbed law remains Gaussian, $M_t^\lambda = \mathcal{N}(\mu_t^\lambda,\Sigma_t^\lambda)$, with $\mu_t^\lambda = (1+\lambda A_t)\mu_t + \lambda B_t$. The moments of the perturbed state law $m_t^\lambda = \mathbb{P}_{X_t^\lambda}$ satisfy
\begin{equation}
\mu_{t+1} = (a+b\theta_t^1+b\theta_t^2+c)\mu_t, \qquad
\Sigma_{t+1} = (a+b\theta_t^1)^2\Sigma_t + (b\theta_t^2+c)^2\lambda^2\rho^2(\mu_t^2+1) + b^2\tau^2+\sigma^2.
\end{equation}


### Closed-form cost
The objective $J^\lambda(\theta)$ admits an explicit expression in terms of $(\mu_t,\Sigma_t)_{t=0}^T$:
\begin{align}
J^\lambda(\theta) = \sum_{t=0}^{T-1}\Big[
& q(\Sigma_t+\mu_t^2) + r\big((\theta_t^1+\theta_t^2)^2\mu_t^2+(\theta_t^1)^2\Sigma_t+(\theta_t^2)^2\lambda^2\rho^2(\mu_t^2+1)+\tau^2\big) \\
& + \gamma\big(\mu_t^2+\lambda^2\rho^2(\mu_t^2+1)\big)\Big] + q_T(\Sigma_T+\mu_T^2) + \gamma_T\big(\mu_T^2+\lambda^2\rho^2(\mu_T^2+1)\big),
\end{align}
which provides ground-truth values $J^\lambda(\theta)$ for any $\theta$ and $\lambda$.

### Optimal Policy
t $\lambda=0$, the optimizer $\theta^\star$ is available via two decoupled Riccati recursions, $P_T=q_T$, $S_T=q_T+\gamma_T$,
\begin{equation}
P_t = q+\frac{a^2rP_{t+1}}{r+b^2P_{t+1}}, \qquad
S_t = q+\gamma+\frac{(a+c)^2rS_{t+1}}{r+b^2S_{t+1}},
\end{equation}
giving
\begin{equation}
\theta_t^{1,\star} = -\frac{abP_{t+1}}{r+b^2P_{t+1}}, \qquad
\theta_t^{2,\star} = \frac{abP_{t+1}}{r+b^2P_{t+1}} - \frac{b(a+c)S_{t+1}}{r+b^2S_{t+1}}.
\end{equation}

### Exact Gradient (oracle)
For any perturbation level $\lambda \geq 0$, an exact $O(T)$ gradient $\nabla_\theta J^\lambda(\theta)$ can be computed by a forward-backward adjoint sweep on the moment dynamics $(\mu_t,\Sigma_t)_{t=0}^T$. Write $D=\lambda^2\rho^2$, $A_t=a+b\theta_t^1+b\theta_t^2+c$, $F_t=a+b\theta_t^1$, and $G_t=b\theta_t^2+c$. The adjoint variables satisfy
\begin{equation}
\lambda_t^\mu = 2\mu_t\Big[q+\gamma+r(\theta_t^1+\theta_t^2)^2+D\big(\gamma+r(\theta_t^2)^2\big)+DG_t^2\lambda_{t+1}^\Sigma\Big] + A_t\lambda_{t+1}^\mu, \qquad
\lambda_t^\Sigma = q+r(\theta_t^1)^2 + F_t^2\lambda_{t+1}^\Sigma,
\end{equation}
terminated by $\lambda_T^\mu=2\big[q_T+\gamma_T(1+D)\big]\mu_T$ and $\lambda_T^\Sigma=q_T$. For each $t=0,\ldots,T-1$, the two components of the exact policy gradient are
\begin{align}
\frac{\partial J^\lambda}{\partial\theta_t^1}
={}&2r\theta_t^1\Sigma_t
+2r(\theta_t^1+\theta_t^2)\mu_t^2
+b\mu_t\lambda_{t+1}^\mu
+2bF_t\Sigma_t\lambda_{t+1}^\Sigma,\\
\frac{\partial J^\lambda}{\partial\theta_t^2}
={}&2r(\theta_t^1+\theta_t^2)\mu_t^2
+2rD\theta_t^2(\mu_t^2+1)
+b\mu_t\lambda_{t+1}^\mu
+2bDG_t(\mu_t^2+1)\lambda_{t+1}^\Sigma.
\end{align}
Hence $\nabla_\theta J^\lambda(\theta)$ is evaluated exactly using one forward moment recursion and one backward adjoint recursion, without Monte Carlo sampling or automatic differentiation. This benchmark therefore provides, for each $(\theta,\lambda)$: (i) the closed-form objective $J^\lambda(\theta)$, (ii) the closed-form optimum $\theta^\star$ at $\lambda=0$, and (iii) the exact gradient $\nabla_\theta J^\lambda(\theta)$, against which the proposed estimators are compared.


In [1]:
import numpy as np
import torch
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

set_seed(0)
print(device)

cuda


In [ ]:
# System parameters
T = 10                  # Time horizon
a, b, c = 1.0, 0.5, 0.2 # Dynamics coefficients
sigma = 0.1             # Environment noise std
tau = 0.1               # Policy noise std
mu_0 = 2.0              # Initial state mean
std_0 = 0.5             # Initial state std

# Cost parameters
q_cost = 1.0            # State cost
r_cost = 0.1            # Action cost
gamma_cost = 1.0        # Mean-field measure cost
q_T = 2.0               # Terminal state cost
gamma_T = 2.0           # Terminal mean-field measure cost

# Perturbation parameters
lambdas = [0, 0.5]      # Perturbation scale
rho = 1.0               # Exogenous noise std


## Closed-form objective, optimum, and gradient

In [3]:
# Exact theta, moment, cost and gradient computations, with finite difference gradient for verification
def moment_recursions(theta1, theta2, lam):
    dtype = theta1.dtype
    mu = torch.as_tensor(mu_0, dtype=dtype, device=device)
    var = torch.as_tensor(std_0 ** 2, dtype=dtype, device=device)
    mus = [mu]
    vars_ = [var]
    var_mu_l = []
    for t in range(T):
        vmu = (lam ** 2) * (rho ** 2) * (mu ** 2 + 1.0)
        A = a + b * theta1[t] + b * theta2[t] + c
        F = a + b * theta1[t]
        G = b * theta2[t] + c
        mu = A * mu
        var = F ** 2 * var + G ** 2 * vmu + b ** 2 * tau ** 2 + sigma ** 2
        mus.append(mu)
        vars_.append(var)
        var_mu_l.append(vmu)
    return torch.stack(mus), torch.stack(vars_), torch.stack(var_mu_l)


def exact_cost(theta1, theta2, lam):
    mus, vars_, var_mu_l = moment_recursions(theta1, theta2, lam)
    total = torch.zeros((), dtype=theta1.dtype, device=device)
    for t in range(T):
        mu_t = mus[t]
        var_t = vars_[t]
        vmu_t = var_mu_l[t]
        e_x2 = var_t + mu_t ** 2
        e_mu_l2 = mu_t ** 2 + vmu_t
        e_a2 = (theta1[t] + theta2[t]) ** 2 * mu_t ** 2 + theta1[t] ** 2 * var_t + theta2[t] ** 2 * vmu_t + tau ** 2
        total = total + q_cost * e_x2 + r_cost * e_a2 + gamma_cost * e_mu_l2
    mu_T = mus[-1]
    var_T = vars_[-1]
    vmu_T = (lam ** 2) * (rho ** 2) * (mu_T ** 2 + 1.0)
    total = total + q_T * (var_T + mu_T ** 2) + gamma_T * (mu_T ** 2 + vmu_T)
    return total


def exact_policy_gradient(theta1, theta2, lam):
    theta1 = torch.as_tensor(theta1, dtype=DTYPE, device=device)
    theta2 = torch.as_tensor(theta2, dtype=DTYPE, device=device)
    mus, vars_, _ = moment_recursions(theta1, theta2, lam)
    D = (lam ** 2) * (rho ** 2)

    adj_mu = torch.empty(T + 1, dtype=DTYPE, device=device)
    adj_var = torch.empty(T + 1, dtype=DTYPE, device=device)
    adj_mu[T] = 2.0 * (q_T + gamma_T * (1.0 + D)) * mus[T]
    adj_var[T] = q_T

    for t in range(T - 1, -1, -1):
        H = a + b * theta1[t] + b * theta2[t] + c
        F = a + b * theta1[t]
        G = b * theta2[t] + c
        mean_weight = (
            q_cost
            + gamma_cost
            + r_cost * (theta1[t] + theta2[t]) ** 2
            + D * (gamma_cost + r_cost * theta2[t] ** 2)
        )
        adj_mu[t] = (
            2.0 * mus[t] * (mean_weight + D * G ** 2 * adj_var[t + 1])
            + H * adj_mu[t + 1]
        )
        adj_var[t] = (
            q_cost
            + r_cost * theta1[t] ** 2
            + F ** 2 * adj_var[t + 1]
        )

    grad1 = torch.empty(T, dtype=DTYPE, device=device)
    grad2 = torch.empty(T, dtype=DTYPE, device=device)
    for t in range(T):
        F = a + b * theta1[t]
        G = b * theta2[t] + c
        perturbed_mean_var = D * (mus[t] ** 2 + 1.0)
        common_mean = (
            2.0 * r_cost * (theta1[t] + theta2[t]) * mus[t] ** 2
            + b * mus[t] * adj_mu[t + 1]
        )
        grad1[t] = (
            2.0 * r_cost * theta1[t] * vars_[t]
            + common_mean
            + 2.0 * b * F * vars_[t] * adj_var[t + 1]
        )
        grad2[t] = (
            common_mean
            + 2.0 * r_cost * theta2[t] * perturbed_mean_var
            + 2.0 * b * G * perturbed_mean_var * adj_var[t + 1]
        )
    return grad1, grad2


def exact_theta():
    P = torch.zeros(T + 1, dtype=DTYPE, device=device)
    S = torch.zeros(T + 1, dtype=DTYPE, device=device)
    P[T] = q_T
    S[T] = q_T + gamma_T
    theta1 = torch.zeros(T, dtype=DTYPE, device=device)
    theta2 = torch.zeros(T, dtype=DTYPE, device=device)
    for t in reversed(range(T)):
        theta1[t] = -(a * b * P[t + 1]) / (r_cost + b ** 2 * P[t + 1])
        L_t = -(b * (a + c) * S[t + 1]) / (r_cost + b ** 2 * S[t + 1])
        theta2[t] = L_t - theta1[t]
        P[t] = q_cost + (a ** 2 * r_cost * P[t + 1]) / (r_cost + b ** 2 * P[t + 1])
        S[t] = q_cost + gamma_cost + ((a + c) ** 2 * r_cost * S[t + 1]) / (r_cost + b ** 2 * S[t + 1])
    return theta1, theta2, P, S


def finite_difference_gradient(theta1, theta2, lam, eps=1e-6):
    grad1 = torch.empty_like(theta1)
    grad2 = torch.empty_like(theta2)

    for t in range(T):
        step = torch.zeros(T, dtype=DTYPE, device=device)
        step[t] = eps
        grad1[t] = (
            exact_cost(theta1 + step, theta2, lam)
            - exact_cost(theta1 - step, theta2, lam)
        ) / (2.0 * eps)
        grad2[t] = (
            exact_cost(theta1, theta2 + step, lam)
            - exact_cost(theta1, theta2 - step, lam)
        ) / (2.0 * eps)

    return grad1, grad2

In [6]:
benchmarks = {}
theta1, theta2, _, _ = exact_theta()

for lam in lambdas:
    cost = exact_cost(theta1, theta2, lam)
    grad1, grad2 = exact_policy_gradient(theta1, theta2, lam)
    grad1_fd, grad2_fd = finite_difference_gradient(theta1, theta2, lam)

    error1 = torch.max(torch.abs(grad1 - grad1_fd)).item()
    error2 = torch.max(torch.abs(grad2 - grad2_fd)).item()
    max_fd_error = max(error1, error2)

    benchmarks[(lam, "exact")] = {
        "cost": cost.item(),
        "theta1": theta1,
        "theta2": theta2,
        "grad1": grad1,
        "grad2": grad2
    }

    print(f"\nWith lambda={lam}:")
    print(f"Exact cost:", cost.item())
    print(f"Exact policy gradient norms:", torch.norm(grad1).item(), torch.norm(grad2).item())
    print(f"Maximum finite-difference error: {max_fd_error:.3e}")




With lambda=0:
Exact cost: 10.494682973131821
Exact policy gradient norms: 4.7372563526178795e-17 3.469514780246362e-17
Maximum finite-difference error: 3.553e-09

With lambda=0.5:
Exact cost: 14.6459267535937
Exact policy gradient norms: 0.17643877732611626 0.16541724230322427
Maximum finite-difference error: 3.624e-09


## REINFORCE Implementation

In [50]:
def mean_flow(theta1, theta2):
    mu = torch.as_tensor(mu_0, dtype=DTYPE, device=device)
    mus = [mu]
    for t in range(T):
        closed_loop_mean = a + b * theta1[t] + b * theta2[t] + c
        mu = closed_loop_mean * mu
        mus.append(mu)
    return torch.stack(mus).detach()


def sample_batch(theta1, theta2, lam, n=10000, seed=None, frozen_mu_flow=None):
    if seed is not None:
        torch.manual_seed(seed)

    # Deterministic mean flow
    if frozen_mu_flow is None: mu_flow = mean_flow(theta1, theta2)
    else: mu_flow = frozen_mu_flow.detach()

    # Sample trajectories for the randomized-law LQ system
    states = torch.empty((n, T + 1), dtype=DTYPE, device=device)
    actions = torch.empty((n, T), dtype=DTYPE, device=device)
    action_means = torch.empty((n, T), dtype=DTYPE, device=device)
    policy_noises = torch.empty((n, T), dtype=DTYPE, device=device)
    law_means = torch.empty((n, T + 1), dtype=DTYPE, device=device)
    stage_costs = torch.empty((n, T), dtype=DTYPE, device=device)

    x = mu_0 + std_0 * torch.randn(n, dtype=DTYPE, device=device)
    states[:, 0] = x

    with torch.no_grad():
        for t in range(T):
            perturb_A = rho * torch.randn(n, dtype=DTYPE, device=device)
            perturb_B = rho * torch.randn(n, dtype=DTYPE, device=device)
            mu_law = (1.0 + lam * perturb_A) * mu_flow[t] + lam * perturb_B
            law_means[:, t] = mu_law

            eta = torch.randn(n, dtype=DTYPE, device=device)
            mean_action = theta1[t] * x + theta2[t] * mu_law
            action = mean_action + tau * eta

            stage_cost = q_cost * x ** 2 + r_cost * action ** 2 + gamma_cost * mu_law ** 2
            
            eps = torch.randn(n, dtype=DTYPE, device=device)
            x_next = a * x + b * action + c * mu_law + sigma * eps

            actions[:, t] = action
            action_means[:, t] = mean_action
            policy_noises[:, t] = eta
            stage_costs[:, t] = stage_cost
            states[:, t + 1] = x_next
            x = x_next
        
        perturb_A_T = rho * torch.randn(n, dtype=DTYPE, device=device)
        perturb_B_T = rho * torch.randn(n, dtype=DTYPE, device=device)
        mu_law_T = (1.0 + lam * perturb_A_T) * mu_flow[T] + lam * perturb_B_T
        law_means[:, T] = mu_law_T

        terminal_costs = q_T * states[:, T] ** 2 + gamma_T * mu_law_T ** 2

    return {
        "states": states,
        "actions": actions,
        "action_means": action_means,
        "policy_noises": policy_noises,
        "law_means": law_means,
        "stage_costs": stage_costs,
        "terminal_costs": terminal_costs,
        "mu_flow": mu_flow,
    }


def reinforce_gradient(theta1, theta2, lam, n=10000, use_baseline=True, seed=None):
    batch = sample_batch(theta1, theta2, lam, n, seed=seed)

    states = batch["states"]
    actions = batch["actions"]
    action_means = batch["action_means"]
    law_means = batch["law_means"]
    stage_costs = batch["stage_costs"]
    terminal_costs = batch["terminal_costs"]

    # returns-to-go
    returns = torch.empty((stage_costs.shape[0], T), dtype=DTYPE, device=device)
    running = terminal_costs.clone()
    for t in reversed(range(T)):
        running = stage_costs[:, t] + running
        returns[:, t] = running
    weights = returns

    if use_baseline:
        weights = weights - weights.mean(dim=0, keepdim=True)
    
    score_common = (actions - action_means) / (tau ** 2)
    score_theta1 = score_common * states[:, :T]
    score_theta2 = score_common * law_means[:, :T]

    grad1 = torch.mean(score_theta1 * weights, dim=0)
    grad2 = torch.mean(score_theta2 * weights, dim=0)

    return grad1.detach(), grad2.detach()


def average_reinforce(theta1, theta2, lam, n=15000, n_seeds=20):
    gs1, gs2 = [], []
    for seed in range(n_seeds):
        g1, g2 = reinforce_gradient(
            theta1, theta2, lam, n=n, use_baseline=True, seed=seed
        )
        gs1.append(g1)
        gs2.append(g2)
    return torch.stack(gs1).mean(0), torch.stack(gs2).mean(0), torch.stack(gs1).std(0), torch.stack(gs2).std(0)


def monte_carlo_cost(theta1, theta2, lam, n=10000, seed=None):
    batch = sample_batch(theta1, theta2, lam, n, seed=seed)
    total_costs = batch["stage_costs"].sum(dim=1) + batch["terminal_costs"]
    return total_costs.mean().detach()


def frozen_monte_carlo_cost(theta1, theta2, lam, frozen_mu_flow, n=100000, seed=None):
    batch = sample_batch(theta1, theta2, lam, n=n, seed=seed, frozen_mu_flow=frozen_mu_flow)
    total_costs = batch["stage_costs"].sum(dim=1) + batch["terminal_costs"]
    return total_costs.mean()


def frozen_flow_finite_difference(theta1, theta2, lam, eps=1e-4, n=200000, seed=123):
    frozen_mu = mean_flow(theta1, theta2)

    g1 = torch.zeros_like(theta1)
    g2 = torch.zeros_like(theta2)

    for t in range(T):
        e1 = torch.zeros_like(theta1)
        e1[t] = eps

        jp = frozen_monte_carlo_cost(theta1 + e1, theta2, lam, frozen_mu, n=n, seed=seed)
        jm = frozen_monte_carlo_cost(theta1 - e1, theta2, lam, frozen_mu, n=n, seed=seed)
        g1[t] = (jp - jm) / (2 * eps)

        e2 = torch.zeros_like(theta2)
        e2[t] = eps

        jp = frozen_monte_carlo_cost(theta1, theta2 + e2, lam, frozen_mu, n=n, seed=seed)
        jm = frozen_monte_carlo_cost(theta1, theta2 - e2, lam, frozen_mu, n=n, seed=seed)
        g2[t] = (jp - jm) / (2 * eps)

    return g1, g2

In [51]:


for lam in lambdas:
    reinf_g1, reinf_g2, reinf_g1_std, reinf_g2_std = average_reinforce(theta1, theta2, lam, n=15_000, n_seeds=20)
    mc_cost = monte_carlo_cost(theta1, theta2, lam, n=15_000)

    theta1_noise = theta1 + 0.01 * torch.randn_like(theta1)
    theta2_noise = theta2 + 0.01 * torch.randn_like(theta2)
    reinf_g1_noise, reinf_g2_noise, _, _ = average_reinforce(theta1_noise, theta2_noise, lam, n=15_000, n_seeds=20)
    exact_g1_noise, exact_g2_noise = exact_policy_gradient(theta1_noise, theta2_noise, lam)

    ff_g1, ff_g2 = frozen_flow_finite_difference(theta1, theta2, lam, eps=1e-4, n=200_000, seed=123)

    print(f"\nWith lambda={lam}:")
    def cosine(g_hat, g):
        return torch.dot(g_hat.flatten(), g.flatten()) / (
            torch.norm(g_hat) * torch.norm(g) + 1e-12
        )

    print("cos theta1:", cosine(reinf_g1_noise, exact_g1_noise).item())
    print("cos theta2:", cosine(reinf_g2_noise, exact_g2_noise).item())

    print(f"REINFORCE gradient stds:", reinf_g1_std.mean().item(), reinf_g2_std.mean().item())
    
    print(f"REINFORCE gradient norms:", torch.norm(reinf_g1).item(), torch.norm(reinf_g2).item())
    print(f"Exact gradient norms: ", torch.norm(benchmarks[(lam, "exact")]["grad1"]).item(), torch.norm(benchmarks[(lam, "exact")]["grad2"]).item())
    print(f"REINFORCE vs exact diff:", torch.norm(reinf_g1 - benchmarks[(lam, "exact")]["grad1"]).item(), torch.norm(reinf_g2 - benchmarks[(lam, "exact")]["grad2"]).item())

    print(f"REINFORCE gradient norms at perturbed theta:", torch.norm(reinf_g1_noise).item(), torch.norm(reinf_g2_noise).item())
    print(f"Exact gradient norms at perturbed theta:", torch.norm(exact_g1_noise).item(), torch.norm(exact_g2_noise).item())
    print(f"REINFORCE vs exact diff at perturbed theta:", torch.norm(reinf_g1_noise - exact_g1_noise).item(), torch.norm(reinf_g2_noise - exact_g2_noise).item())
    print(f"REINFORCE vs exact relative diff at perturbed theta:", torch.norm(reinf_g1_noise - exact_g1_noise).item() / torch.norm(exact_g1_noise).item(), torch.norm(reinf_g2_noise - exact_g2_noise).item() / torch.norm(exact_g2_noise).item())

    print(f"Frozen flow finite difference gradient norms:", torch.norm(ff_g1).item(), torch.norm(ff_g2).item())
    print("REINFORCE vs frozen-flow FD diff:",
      torch.norm(reinf_g1 - ff_g1).item(),
      torch.norm(reinf_g2 - ff_g2).item())
    print("REINFORCE vs frozen-flow FD cosine:",
      cosine(reinf_g1, ff_g1).item(),
      cosine(reinf_g2, ff_g2).item())
    print("REINFORCE vs frozen-flow FD relative diff:",
      torch.norm(reinf_g1 - ff_g1).item() / (torch.norm(ff_g1).item() + 1e-12),
      torch.norm(reinf_g2 - ff_g2).item() / (torch.norm(ff_g2).item() + 1e-12))
    
    print(f"Monte Carlo cost:", mc_cost.item())
    print(f"Exact cost:      ", exact_cost(theta1, theta2, lam).item())


With lambda=0:
cos theta1: -0.999821521241843
cos theta2: -0.999925761611809
REINFORCE gradient stds: 0.06858359763341708 0.06472032054783626
REINFORCE gradient norms: 0.5925536064152694 0.6638738096128679
Exact gradient norms:  4.7372563526178795e-17 3.469514780246362e-17
REINFORCE vs exact diff: 0.5925536064152694 0.6638738096128679
REINFORCE gradient norms at perturbed theta: 0.5131263031033823 0.5862811250745156
Exact gradient norms at perturbed theta: 0.12811809915445138 0.1262980589192544
REINFORCE vs exact diff at perturbed theta: 0.6412261042473608 0.7125714696273542
REINFORCE vs exact relative diff at perturbed theta: 5.004961113841828 5.641982748784123
Frozen flow finite difference gradient norms: 0.7475917676449342 0.7472096641404425
REINFORCE vs frozen-flow FD diff: 0.15509655861307797 0.08335198486497311
REINFORCE vs frozen-flow FD cosine: 0.9999795580805266 0.9999972898727667
REINFORCE vs frozen-flow FD relative diff: 0.20746156569039834 0.11155099949188432
Monte Carlo c

The above simulation shows that:
1. The cost simulation using REINFORCE is correct: $$\hat{J}^\lambda (\theta) \approx J^\lambda (\theta).$$
2. Standard REINFORCE is implemented correctly: $$\hat{g}_\text{REINFORCE} \approx \nabla J^\lambda_\text{frozen}(\theta).$$
3. Frozen-flow gradient differs from full mean-field gradient: $$\nabla J^\lambda_\text{frozen} (\theta) \neq \nabla J^\lambda (\theta).$$

Where we define $J^\lambda_\text{frozen}(\theta;\theta_0)$ as the cost in which the policy uses $\theta$, but the law input uses the frozen flow $\mu_t^{\theta_0}$.

The standard REINFORCE does not solve the full mean-field control problem. It solves a surrogate problem where the population flow is treated as fixed while differentiating the policy parameters.